# Tic Tac Toe Reinforcement Learning (Q-Learning)

This project implements a custom Tic Tac Toe environment and a Q-Learning bot that learns optimal strategies by playing against itself or humans.

In [1]:
import random as rd

In [2]:
a =rd.random()

In [3]:
class TicTacToe:
    def __init__(self):
        self.board=["0"]*9
        self.counter=1
    def getstate(self):
        return  self.board

    def reset (self):
        self.board=["0"]*9
        self.counter=1

    def solvecheck(self):
        winning_combinations = [
            (0, 1, 2),
            (3, 4, 5),
            (6, 7, 8),
            (0, 3, 6),
            (1, 4, 7),
            (2, 5, 8),
            (0, 4, 8),
            (2, 4, 6) ] 
        solved=False
        for a, b, c in winning_combinations:
            if self.board[a] != "0" and self.board[a] ==self.board[b] == self.board[c]:
                solved=True
                return self.board[a]
        if solved==False:
            return "0"
                
    def printboard(self):
        for i in range (9):
            print(self.board[i] , end ="")
            print(" | ", end ="")
            if i == 2:
                print("")
                print("____________")
            if i == 5:
                print("")
                print("____________")
            if i == 8:
                print("")

    def legalmoves(self):
        l_moves=[]

        for i, item in enumerate(self.board): 
            if item == "0":
                l_moves.append(i)
        return l_moves

    def move(self,action,player):
        if self.board[action]=="0":
            self.board[action]=player
            self.counter=self.counter + 1
        else :
            print("not legal move")
            available=self.legalmoves()
            print(f"try something from these  {available}")

    def playerturnchecker(self):
        if self.counter%2==0:
            return "o"
        else :
            return "x"
            
    def play(self):
        playing=True
        while playing:
            self.printboard()
            turn = self.playerturnchecker()
            print (f"Player {turn} move ")
            print(self.legalmoves())
            if len(self.legalmoves())!=0:
                action=int(input("enter move number from legal moves "))
                self.move(action,turn)
                check=self.solvecheck()
                if check!="0":
                    self.printboard()
                    print(f"Player {check} wins !")
                    a=input("u want to restart?y/n")
                    if (a== "n"):
                        playing = False
                    else:
                        self.reset()
                        print("Game restsart")
            else : 
                print ("Match is Draw!")
                a=input("u want to restart?y/n")
                if (a== "n"):
                    print("Game end")
                    playing = False
                else:
                    self.reset()
                    print("Game restsart")
    
    
        
    def playwithbot(self, bot):

        bot.e = 0
        playing = True
    
        self.reset()
    
        while playing:
    
            self.printboard()
    
            turn = self.playerturnchecker()
    
            # ================= HUMAN TURN =================
            if turn == "x":
    
                legal_actions = self.legalmoves()
    
                print("Your move")
                print("Legal moves:", legal_actions)
    
                action = int(input("Enter move number: "))
    
                if action not in legal_actions:
                    print("Invalid move")
                    continue
    
                self.move(action, "x")

                # Check if HUMAN won
                check = self.solvecheck()
    
                if check == "x":
    
                    self.printboard()
                    print("You win!")
    
                    a = input("You want to restart? y/n: ")
    
                    if a == "n":
                        playing = False
                    else:
                        self.reset()
    
                    continue
    
                # Check DRAW after human move
                if len(self.legalmoves()) == 0:
    
                    self.printboard()
                    print("Match is Draw!")
    
                    a = input("You want to restart? y/n: ")
    
                    if a == "n":
                        playing = False
                    else:
                        self.reset()
    
                    continue

            # ================= AI TURN =================
            else:
    
                print("AI is thinking...")
    
                state = self.getstate().copy()
                legal_actions = self.legalmoves()
    
                # Safety check
                if len(legal_actions) == 0:
                    print("Match is Draw!")
                    break
    
                action = bot.action(state, legal_actions)
    
                print("AI chooses:", action)
    
                self.move(action, "o")
    
                # Check if AI won
                check = self.solvecheck()
    
                if check == "o":
    
                    self.printboard()
                    print("AI wins!")
    
                    a = input("You want to restart? y/n: ")
    
                    if a == "n":
                        playing = False
                    else:
                        self.reset()
    
                    continue

                # Check DRAW after AI move
                if len(self.legalmoves()) == 0:
    
                    self.printboard()
                    print("Match is Draw!")
    
                    a = input("You want to restart? y/n: ")
    
                    if a == "n":
                        playing = False
                    else:
                        self.reset()
    
                    continue


    def trainwithpercentage(self, bot_x, bot_o, episodes):

        import time
    
        start_time = time.time()
    
        # Print progress every this many episodes
        progress_interval = 100000
    
        for episode in range(episodes):
    
            self.reset()
    
            done = False
    
            while not done:
    
                turn = self.playerturnchecker()
    
                # ================= AI X =================
                if turn == "x":
    
                    state = self.getstate().copy()
                    legal_actions = self.legalmoves()

                    action = bot_x.action(state, legal_actions)
    
                    self.move(action, "x")
    
                    next_state = self.getstate().copy()
    
                    check = self.solvecheck()
    
                    # X wins
                    if check == "x":
    
                        bot_x.update_q(
                            state,
                            action,
                            1,
                            next_state,
                            [],
                            True
                        )
    
                        if bot_o.s_a_history:
    
                            old_state, old_action = bot_o.s_a_history[-1]
    
                            bot_o.update_q(
                                old_state,
                                old_action,
                                -1,
                                next_state,
                                [],
                                True
                            )
    
                        done = True
    
                    # Draw
                    elif len(self.legalmoves()) == 0:

                        bot_x.update_q(
                            state,
                            action,
                            0,
                            next_state,
                            [],
                            True
                        )
    
                        if bot_o.s_a_history:
    
                            old_state, old_action = bot_o.s_a_history[-1]
    
                            bot_o.update_q(
                                old_state,
                                old_action,
                                0,
                                next_state,
                                [],
                                True
                            )
    
                        done = True
    
                    # Game continues
                    else:
    
                        next_legal_actions = self.legalmoves()
    
                        bot_x.update_q(
                            state,
                            action,
                            0,
                            next_state,
                            next_legal_actions,
                            False
                        )

                # ================= AI O =================
                else:
    
                    state = self.getstate().copy()
                    legal_actions = self.legalmoves()
    
                    action = bot_o.action(state, legal_actions)
    
                    self.move(action, "o")
    
                    next_state = self.getstate().copy()
    
                    check = self.solvecheck()
    
                    # O wins
                    if check == "o":
    
                        bot_o.update_q(
                            state,
                            action,
                            1,
                            next_state,
                            [],
                            True
                        )
    
                        if bot_x.s_a_history:
    
                            old_state, old_action = bot_x.s_a_history[-1]
    
                            bot_x.update_q(
                                old_state,
                                old_action,
                                -1,
                                next_state,
                                [],
                                True
                            )

                        done = True
    
                    # Draw
                    elif len(self.legalmoves()) == 0:
    
                        bot_o.update_q(
                            state,
                            action,
                            0,
                            next_state,
                            [],
                            True
                        )
    
                        if bot_x.s_a_history:
    
                            old_state, old_action = bot_x.s_a_history[-1]
    
                            bot_x.update_q(
                                old_state,
                                old_action,
                                0,
                                next_state,
                                [],
                                True
                            )
    
                        done = True
    
                    # Game continues
                    else:
    
                        next_legal_actions = self.legalmoves()
    
                        bot_o.update_q(
                            state,
                            action,
                            0,
                            next_state,
                            next_legal_actions,
                            False
                        )
    
            # Clear history for next game
            bot_x.s_a_history = []
            bot_o.s_a_history = []
    
            # ================= PROGRESS =================
            if (episode + 1) % progress_interval == 0:
    
                elapsed = time.time() - start_time
    
                completed = episode + 1
                remaining = episodes - completed
    
                episodes_per_second = completed / elapsed
    
                estimated_remaining = remaining / episodes_per_second
    
                percentage = (completed / episodes) * 100
    
                print(
                    f"Progress: {completed:,}/{episodes:,} "
                    f"({percentage:.1f}%) | "
                    f"Time: {elapsed:.1f}s | "
                    f"Speed: {episodes_per_second:,.0f} episodes/s | "
                    f"ETA: {estimated_remaining:.1f}s"
                )
    
        total_time = time.time() - start_time
    
        print("\nTraining complete")
        print(f"Total episodes: {episodes:,}")
        print(f"Total time: {total_time:.2f} seconds")
        print(f"Average speed: {episodes / total_time:,.0f} episodes/s")


    

In [4]:
class ai :
    def __init__(self):
        self.q_table={}
        self.s_a_history=[]
        self.alpha=0.1
        self.gamma=0.9
        self.e=0.5
    def action(self,state,l_actions):
        state_str="".join(state)
        if state_str not in self.q_table:
            self.q_table[state_str] = {}
            for i in l_actions:
                self.q_table[state_str][i]=0
        if rd.random() < self.e :
            a = rd.choice(l_actions)
            self.s_a_history.append([state_str,a])
            return a
        else:
            rewards_list=[]
            for i in l_actions:
                    rewards_list.append(self.q_table[state_str][i])
            a=max(rewards_list)
            a_i=rewards_list.index(a)
            self.s_a_history.append([state_str,l_actions[a_i]])
            return l_actions[a_i]
    def target(self, next_state, next_legal_actions, reward):
        state_str_next = "".join(next_state)
    
        if state_str_next not in self.q_table:
            self.q_table[state_str_next] = {}
    
            for i in next_legal_actions:
                self.q_table[state_str_next][i] = 0
    
        rewards_list = []
    
        for i in next_legal_actions:
            rewards_list.append(self.q_table[state_str_next][i])
    
        max_future_q = max(rewards_list)
    
        return reward + self.gamma * max_future_q


    def update_q(self, state, action, reward, next_state, next_legal_actions, done):
    
        state_str = "".join(state)
    
        old_q = self.q_table[state_str][action]
    
        if done:
            target = reward
    
        else:
            target = self.target(
                next_state,
                next_legal_actions,
                reward
            )
    
        new_q = old_q + self.alpha * (target - old_q)
    
        self.q_table[state_str][action] = new_q
   
    
                   
        

In [5]:
import pickle

with open("q_tableo.pkl", "rb") as file:
    q_table = pickle.load(file)

print(type(q_table))

<class 'dict'>


In [6]:
game = TicTacToe()

bot_o = ai()
bot_o.q_table=q_table


In [7]:
game.playwithbot(bot_o)


0 | 0 | 0 | 
____________
0 | 0 | 0 | 
____________
0 | 0 | 0 | 
Your move
Legal moves: [0, 1, 2, 3, 4, 5, 6, 7, 8]


Enter move number:  0


x | 0 | 0 | 
____________
0 | 0 | 0 | 
____________
0 | 0 | 0 | 
AI is thinking...
AI chooses: 4
x | 0 | 0 | 
____________
0 | o | 0 | 
____________
0 | 0 | 0 | 
Your move
Legal moves: [1, 2, 3, 5, 6, 7, 8]


Enter move number:  4


Invalid move
x | 0 | 0 | 
____________
0 | o | 0 | 
____________
0 | 0 | 0 | 
Your move
Legal moves: [1, 2, 3, 5, 6, 7, 8]


Enter move number:  3


... (4 outputs left out)

x | o | x | 
____________
x | o | 0 | 
____________
o | x | 0 | 
AI is thinking...
AI chooses: 5
x | o | x | 
____________
x | o | o | 
____________
o | x | 0 | 
Your move
Legal moves: [8]


Enter move number:  8


x | o | x | 
____________
x | o | o | 
____________
o | x | x | 
Match is Draw!


You want to restart? y/n:  n


In [8]:
q_table

{'x00000000': {1: -0.016781273278819793,
  2: -0.003506726257158297,
  3: -0.003814202508313043,
  4: 0.4118457927842995,
  5: 0.0,
  6: 0.0,
  7: 0.0,
  8: 0.0},
 'xo0000000': {2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0},
 'xo000000x': {2: -0.37221488370823236,
  3: -0.37899499789690066,
  4: 0.14373182699999995,
  5: -0.428612753062134,
  6: -0.39630037632675824,
  7: -0.38417064395334294},
 'xo0o0000x': {2: 0, 4: 0, 5: 0, 6: 0, 7: 0},
 'xo0o0x00x': {2: -0.34967492841911557,
  4: -0.42609917711008455,
  6: -0.44955853568079357,
  7: -0.4835640871489078},
 'xooo0x00x': {4: 0, 6: 0, 7: 0},
 '00x000000': {0: -0.0001768986334736225,
  1: -0.006285262342669039,
  3: -0.0020744547269343776,
  4: 0.4811320624252072,
  5: 0.0,
  6: 0.0,
  7: 0.0,
  8: 0.0},
 'o0x000000': {1: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0},
 'o0x0000x0': {1: -0.01984386328346864,
  3: -0.016289057638086614,
  4: -0.03522700353850689,
  5: -0.043393180970156736,
  6: 0.4783876540370998,
  8: 0.0},
 'oox0000x0': {3: 0, 4: